<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
RAG-Agent
</b></font> </br></p>

---

In [1]:
#@title 🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M14-RAG-Agent"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 34.142.134.27
Hostname: 27.134.142.34.bc.googleusercontent.com
Stadt: Singapore
Region: Singapore
Land: SG
Koordinaten: 1.2897,103.8501
Provider: AS396982 Google LLC
Postleitzahl: 018989
Zeitzone: Asia/Singapore


In [ ]:
#@title 📦 Pakete installieren{ display-mode: "form" }
install_packages([
                ('markitdown[all]', 'markitdown'),
                'langchain_huggingface',
                ('unstructured[all-docs]>=0.11.2', 'unstructured'),
                ])

# 1 | Übersicht
---



In *RAG-Chain mit LangChain* haben wir eine **RAG-Chain** gebaut: deterministisch, linearer Ablauf.  
Jetzt machen wir den nächsten Schritt: **RAG als Tool für einen Agenten**.

**Von der Chain zum Agent**

| Merkmal | RAG-Chain (M10) | RAG-Agent (M11) |
|---------|----------------|----------------|
| Ablauf | Immer: Retrieve → Generate | Nur bei Bedarf: Agent entscheidet |
| RAG-Nutzung | Immer aktiv | Tool, das der Agent wählt |
| Weiteres Wissen | Nur Vektordatenbank | Eigenes LLM-Wissen + RAG + andere Tools |
| Flexibilität | Niedrig | Hoch |
| Komplexität | Gering | Mittel |

**Was wir bauen**

Ein Agenten-System mit **RAG als einem von mehreren Tools**:
- `suche_wissensdatenbank` – RAG-Tool für spezifische Kursfragen
- `erklaere_konzept` – Direktes LLM-Wissen für allgemeine Erklärungen
- `berechne_tokens` – Einfaches Berechnungstool

In [2]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    U["👤 Nutzer-Anfrage"] --> A["🤖 RAG-Agent"]
    A --> D{"Welches Tool?"}

    D -->|"Spezifische\nKursfrage"| RAG["📚 suche_wissensdatenbank()\n(ChromaDB Retrieval)"]
    D -->|"Allgemeines\nKonzept"| LLM["🧠 erklaere_konzept()\n(LLM-Wissen)"]
    D -->|"Token-\nBerechnung"| CALC["🔢 berechne_tokens()\n(Formel)"]

    RAG --> ANS["💬 Antwort"]
    LLM --> ANS
    CALC --> ANS
'''

mermaid(diagram, width=650)

# 2 | RAG als Tool
---

Ein **RAG-Tool** ist eine normale LangChain-Tool-Funktion, die intern ChromaDB abfragt.

## Design-Prinzipien für RAG-Tools

| Prinzip | Beschreibung |
|---------|-------------|
| **Klarer Name** | Name sagt dem Agenten, wann das Tool zu nutzen ist |
| **Präzise Beschreibung** | Docstring erklärt genau, welche Fragen das Tool beantwortet |
| **Fehlerbehandlung** | Gibt lesbaren Text zurück, kein `raise` |
| **Formatierte Ausgabe** | Gibt Quellen + Antwort zurück |

In [3]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.agents import create_agent

# LLM und Embeddings
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Initialisierung abgeschlossen


In [4]:
# Wissensdatenbank aufbauen (aus M09/M10 bekannt)
texte = [
    ("LangChain 1.0 bietet init_chat_model() fuer einheitliche Modell-Initialisierung. "
     "LCEL-Chains verwenden den Pipe-Operator | zur Verkettung. "
     "create_agent() ist die moderne API fuer Agenten.",
     {"quelle": "langchain", "kategorie": "framework"}),
    ("LangGraph ist ein Framework fuer zustandsbasierte Agenten-Workflows. "
     "StateGraph definiert Knoten und Kanten. "
     "Checkpointing ermoeglicht persistente Agenten-Sessions.",
     {"quelle": "langgraph", "kategorie": "framework"}),
    ("RAG kombiniert Retrieval mit LLM-Generation. "
     "Embeddings wandeln Text in Vektoren um. "
     "ChromaDB speichert diese Vektoren und ermoeglicht Aehnlichkeitssuche.",
     {"quelle": "rag", "kategorie": "technik"}),
    ("LangSmith ist das Observability-Tool fuer LangChain. "
     "Es zeigt Traces, Kosten und Latenz fuer jeden Agent-Run. "
     "LANGSMITH_PROJECT definiert das Projekt fuer Traces.",
     {"quelle": "langsmith", "kategorie": "monitoring"}),
    ("ChromaDB ist eine lokale Vektordatenbank. "
     "persist_directory speichert Daten dauerhaft auf der Festplatte. "
     "similarity_search_with_score gibt Ergebnisse mit Relevanzwert zurueck.",
     {"quelle": "chromadb", "kategorie": "technik"}),
]

dokumente = [Document(page_content=t, metadata=m) for t, m in texte]
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30)
chunks = splitter.split_documents(dokumente)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="kurs_kb",
    persist_directory="chroma_m14"
)
print(f"Wissensdatenbank bereit: {vectorstore._collection.count()} Chunks")

Wissensdatenbank bereit: 5 Chunks


# 3 | RAG-Agent bauen
---



In [5]:
@tool
def suche_wissensdatenbank(frage: str) -> str:
    """Durchsucht die Kurs-Wissensdatenbank nach spezifischen Informationen
    zu LangChain, LangGraph, RAG, ChromaDB und LangSmith.
    Nutze dieses Tool bei Fragen zu Kurs-spezifischen Inhalten und Frameworks.
    """
    try:
        ergebnisse = vectorstore.similarity_search_with_score(frage, k=3)
        if not ergebnisse:
            return "Keine relevanten Informationen in der Wissensdatenbank gefunden."

        antwort_teile = []
        for doc, score in ergebnisse:
            if score < 1.5:  # Nur relevante Ergebnisse (L2-Distanz)
                quelle = doc.metadata.get("quelle", "unbekannt")
                antwort_teile.append(f"[{quelle}] {doc.page_content}")

        if not antwort_teile:
            return "Keine ausreichend relevanten Informationen gefunden."

        return "\n\n".join(antwort_teile)
    except Exception as e:
        return f"Fehler bei der Datenbanksuche: {str(e)}"


@tool
def erklaere_konzept(konzept: str) -> str:
    """Erklaert ein allgemeines KI- oder Informatik-Konzept kurz und verstaendlich.
    Nutze dieses Tool bei allgemeinen Konzeptfragen (z.B. was ist ein Embedding?,
    was ist ein Transformer?) die nicht kurs-spezifisch sind.
    """
    # Dieses Tool nutzt nur LLM-Wissen (kein RAG)
    erklaer_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.2)
    prompt = f"Erklaere das Konzept '{konzept}' in 2-3 Saetzen fuer einen Einsteiger."
    return erklaer_llm.invoke(prompt).content


@tool
def berechne_tokens(text: str, modell: str = "gpt-4o-mini") -> str:
    """Schaetzt die Anzahl der Tokens fuer einen gegebenen Text.
    Nuetzlich um zu verstehen, wie viel ein Text im Kontext kostet.
    """
    # Grobe Schaetzung: 1 Token ~ 4 Zeichen (Englisch), 3 Zeichen (Deutsch)
    zeichen = len(text)
    tokens_grob = zeichen // 3
    kosten_per_1k = 0.00015  # gpt-4o-mini input
    tokens_per_1k = tokens_grob / 1000
    kosten = tokens_per_1k * kosten_per_1k
    return (
        f"Text: {zeichen} Zeichen\n"
        f"Geschaetzte Tokens: ~{tokens_grob} (Faustregel: 3 Zeichen/Token)\n"
        f"Geschaetzte Kosten ({modell}): ~${kosten:.6f}"
    )


tools = [suche_wissensdatenbank, erklaere_konzept, berechne_tokens]
print(f"Tools definiert: {[t.name for t in tools]}")

Tools definiert: ['suche_wissensdatenbank', 'erklaere_konzept', 'berechne_tokens']


In [ ]:
# RAG-Agent erstellen
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m11_rag_agent_system_prompt.md", mode="S")

rag_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    debug=False
)

print("RAG-Agent erstellt")
print(f"Tools: {[t.name for t in tools]}")


# 4 | Wann RAG, wann eigenes Wissen?
---



Der Agent entscheidet selbst, welches Tool er nutzt – aber wie trifft er diese Entscheidung?

**Entscheidungslogik**

Der Agent wählt das Tool basierend auf **drei Faktoren**:

1. **Tool-Name** – Sprechende Namen helfen: `suche_wissensdatenbank` vs. `erklaere_konzept`
2. **Docstring** – Präzise Beschreibung, wann das Tool einzusetzen ist
3. **Kontext der Frage** – Spezifisch (→ RAG) vs. allgemein (→ eigenes Wissen)

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  flowchart</font> </br></p>

diagram = '''
flowchart TD
    FRAGE["❓ Nutzerfrage"]

    FRAGE --> SPEZ{{"Kurs-spezifisch?\n(LangChain, LangGraph,\nRAG, ChromaDB)"}}
    SPEZ -->|"Ja"| RAG["📚 suche_wissensdatenbank()\nChromaDB Retrieval"]
    SPEZ -->|"Nein"| ALLG{{"Allgemeines\nKonzept?"}}

    ALLG -->|"Ja"| LLM["🧠 erklaere_konzept()\nLLM-Wissen"]
    ALLG -->|"Nein"| CALC{{"Berechnung\nnoetig?"}}

    CALC -->|"Ja"| TOK["🔢 berechne_tokens()\nFormel"]
    CALC -->|"Nein"| DIRECT["💬 Direkte Antwort"]

    RAG --> ANS["Antwort"]
    LLM --> ANS
    TOK --> ANS
    DIRECT --> ANS
'''

mermaid(diagram, width=650)

In [ ]:
# Test: Agent zeigt unterschiedliche Tool-Wahl
test_fragen = [
    # Erwartet: suche_wissensdatenbank
    "Welche Parameter gibt es bei init_chat_model() in LangChain?",
    # Erwartet: erklaere_konzept
    "Was ist ein Transformer-Modell in der KI?",
    # Erwartet: berechne_tokens
    "Wie viele Tokens hat dieser Text: Hallo, ich bin ein KI-Agent!",
]

run_cfg = {"run_name": "RAG-Agent-Tool-Wahl", "tags": ["rag-agent", "m14"]}

for frage in test_fragen:
    print(f"Frage: {frage}")
    antwort = rag_agent.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config=run_cfg
    )
    letzte_antwort = antwort["messages"][-1].content
    print(f"Antwort: {letzte_antwort[:200]}...")
    print("-" * 60)
    print()

# 5 | End-to-End Testing
---

Ein gutes RAG-System braucht systematische Tests.  


Wir testen drei Aspekte:

| Testtyp | Was wird geprüft? | Methode |
|---------|-------------------|--------|
| **Retrieval-Test** | Werden relevante Dokumente gefunden? | Manuell prüfen |
| **Antwortqualität** | Sind die Antworten korrekt? | Expected vs. Actual |
| **Tool-Wahl** | Wählt der Agent das richtige Tool? | Trace analysieren |

In [ ]:
# End-to-End Test: Vollstaendiger Durchlauf
test_szenarien = [
    {
        "frage": "Erklaere mir LangGraph und Checkpointing.",
        "erwartetes_tool": "suche_wissensdatenbank",
        "schluesselwoerter": ["LangGraph", "Checkpointing", "StateGraph"]
    },
    {
        "frage": "Was ist der Unterschied zwischen Precision und Recall?",
        "erwartetes_tool": "erklaere_konzept",
        "schluesselwoerter": ["Precision", "Recall"]
    },
]

print("=== End-to-End Test ===\n")
run_cfg = {"run_name": "E2E-Test", "tags": ["test", "m14"]}

for i, szenario in enumerate(test_szenarien):
    frage = szenario["frage"]
    print(f"Test {i+1}: {frage}")

    ergebnis = rag_agent.invoke(
        {"messages": [{"role": "user", "content": frage}]},
        config=run_cfg
    )

    antwort = ergebnis["messages"][-1].content

    # Schluesselwoerter pruefen
    treffer = sum(1 for kw in szenario["schluesselwoerter"]
                 if kw.lower() in antwort.lower())
    gesamt = len(szenario["schluesselwoerter"])

    print(f"  Antwort: {antwort[:150]}...")
    print(f"  Schluesselwoerter: {treffer}/{gesamt} gefunden")
    print(f"  Status: {'PASS' if treffer >= gesamt // 2 else 'FAIL'}")
    print()

In [ ]:
# Zusammenfassung: RAG-Agent vs. RAG-Chain
mprint("""
## Zusammenfassung: Wann was?

| Kriterium | RAG-Chain (M10) | RAG-Agent (M11) |
|-----------|----------------|----------------|
| Einfache Q&A | ✅ Ideal | Overhead |
| Gemischte Fragen | Nur RAG | ✅ Flexibel |
| Mehrere Wissensquellen | Komplex | ✅ Multi-Tool |
| Nachvollziehbarkeit | ✅ Deterministisch | Trace noetig |
| Einsteiger | ✅ Empfohlen | Fortgeschritten |

**Empfehlung:** Mit RAG-Chain starten, bei Bedarf zum RAG-Agenten upgraden.
""")

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M14-RAG-Agent", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


<p><font color='black' size="5">
Biografien-RAG-Agent
</font></p>

Erweitern Sie den RAG-Agenten aus diesem Modul um eine Biografien-Wissensdatenbank.

**Teilaufgaben:**
1. Laden Sie die Biografien-Dateien (aus `../02_daten/01_text/`) und indexieren Sie sie in einer neuen Collection
2. Erstellen Sie ein `suche_biografien`-Tool, das diese Collection abfragt
3. Erstellen Sie einen Agenten mit BEIDEN Tools: `suche_wissensdatenbank` + `suche_biografien`
4. Testen Sie den Agenten mit 3 Fragen: eine über Kurs-Inhalte, eine über Biografien, eine gemischte

**Bonus:** Fügen Sie einen LangSmith-Test hinzu, der die Tool-Wahl des Agenten protokolliert und prüft, ob das richtige Tool gewählt wurde.


---
### 🔍 Selbstcheck mit `assert`

`assert` prüft eine Bedingung — ist sie `False`, stoppt Python mit einem `AssertionError` und zeigt die Fehlermeldung an:

```python
assert bedingung, "Fehlermeldung"

assert 2 + 2 == 4, "Rechnung falsch"    # ✅ kein Fehler
assert len("Hi") > 5, "Text zu kurz"   # ❌ AssertionError: Text zu kurz
```

**So nutzen Sie den Selbstcheck:**
1. Erstellen Sie Ihren Biografien-RAG-Agenten in den Zellen über diesem Block
2. Speichern Sie den Agenten in **`mein_rag_agent`**
3. Führen Sie die Zelle unten aus — beide Wissensbereiche (Kurs-Inhalte & Biografien) werden geprüft


In [ ]:
#@title ✅ Selbstcheck – Biografien-RAG-Agent  { display-mode: "form" }
# ► Speichere deinen RAG-Agenten in 'mein_rag_agent'.

_ag = mein_rag_agent  # ← Variablennamen anpassen

assert hasattr(_ag, "invoke"), \
    "❌ Kein invoke() – wurde der Agent mit create_agent() erstellt?"

for _frage, _label in [
    ("Was ist ein Embedding-Vektor?",  "Kurs-Inhalte"),
    ("Wer war Marie Curie?",           "Biografie"),
]:
    _r    = _ag.invoke({"messages": [{"role": "user", "content": _frage}]})
    _msgs = _r["messages"]
    _tool_calls = [m for m in _msgs if getattr(m, "type", None) == "tool"]
    _antwort    = _msgs[-1].content
    assert len(_antwort) > 20, \
        f"❌ Antwort zu kurz für '{_frage}' ({len(_antwort)} Z.) – prüfe Tools und Retriever"
    print(f"✅ {_label} · {len(_tool_calls)} Tool-Aufruf(e) · {_antwort[:80]}...")
